# 00 · Start here

These notebooks do the same jobs as the two GUIs (`mtinv` for setting up an inversion,
`mtplot` for plotting the result), one step at a time, with the code in view. Use them when a
job needs something the GUI doesn't offer: an unusual refinement, a different way of handling the
ocean, a custom figure. Everything they write is a normal file, so you can do one step in a
notebook and the next in a GUI.

| notebook | what it does | GUI equivalent |
|---|---|---|
| `01_interpolate_edis` | put every station onto one set of periods | setup tab 0 |
| `02_survey_and_topography` | check stations, choose the map projection, get a DEM, put it in the model frame | setup tabs 1-3 |
| `03_mesh_and_receivers` | build the mesh (refinements, topography, ocean) and the receiver file | setup tab 5 |
| `04_refine_inverted_model` | refine a finished model and add stations to it | setup tab 7 |
| `05_plot_model` | plan views, sections and map layers | plotting GUI |
| `06_model_to_geotiff` | depth slices as GeoTIFFs for GIS | plotting GUI, GeoTIFF tab |

**Data selection isn't here on purpose.** Choosing periods, error floors and which data to keep
needs you to look through the data, so do it in `gateway_mt` and write the GoFEM data file there.
Notebook 03 writes the mesh and receivers that go with it.

## How to use a notebook

* **Start Jupyter** from a terminal, in the `mtinv` environment:
  ```
  conda activate mtinv
  cd <this folder>
  jupyter lab
  ```
  It opens in your browser. Double-click a notebook on the left to open it.
* **Run a cell:** click it and press **Shift + Enter**. Run the cells in order, top to bottom.
  `[*]` next to a cell means it's still running; a number means it's done.
* **Change only the Settings cell** (the first code cell of each notebook). Every value there has a
  comment saying what it is and what units it's in. The cells below it read those values.
* **Something went wrong?** Read the last line of the red error text: the notebooks try to say what
  to fix there. Then fix the setting and run the cells again from the top
  (menu *Run → Restart Kernel and Run All Cells*).
* **Save a copy before you change a notebook** for a real project (*File → Save Notebook As*), so
  the original stays as a reference.

## The worked example

Out of the box the notebooks run on a small synthetic survey in `sample_data/`:

* `sample_data/edis/`: 31 stations, 20 km apart, on land beside Monterey Bay, California. The
  coast and a deep submarine canyon are close by, so the example covers the ocean too. The
  *responses* are dummy data copied from a synthetic model (not from this area); only the station
  positions and the real topography and bathymetry matter here.
* `sample_data/edis_infill/`: 9 extra stations, used to show how to add stations to a finished
  model (notebook 04).
* `sample_data/synthetic_model.py`: turns the mesh from notebook 03 into a stand-in "inversion
  result", so notebooks 04-06 have a model before you have a real one.

Notebook 02 downloads the topography (GMRT, free, needs an internet connection). Everything the
notebooks write goes into `work/` next to them, one folder per step. Delete `work/` to start again.

To use your own data, change the paths in each Settings cell.

## Check the environment

This cell imports what the notebooks need and prints the versions. Every line should say `ok`.
If one doesn't, the `mtinv` environment isn't the one running this notebook: choose it with the
kernel name at the top right, or start `jupyter lab` from inside `conda activate mtinv`.

mtpy prints `GDAL_DATA environment variable is not set` and `Ignore GDAL as it is not working`
when it's first imported, in this and other notebooks. Both are harmless.

In [ ]:
import importlib
import sys

print("python", sys.version.split()[0], "at", sys.executable)

import mtinv_gui
from mtinv_gui import config

config.apply_sys_paths()            # dev paths from ~/.mtinv_gui/settings.json, if any are set
config.ensure_mtpy_numpy_compat()   # mtpy 1.x needs this before it's imported

for name in ["numpy", "scipy", "matplotlib", "pyproj", "shapely", "rasterio", "geopandas",
             "pyvista", "PyDealII.Release", "gofem.mesh_utils", "mtpy"]:
    try:
        mod = importlib.import_module(name)
        version = getattr(mod, "__version__", "")
        print(f"ok  {name} {version}")
    except Exception as exc:
        print(f"MISSING  {name}: {exc}")

print(f"ok  mtinv_gui {mtinv_gui.__version__}")

## Conventions worth knowing

**The model frame.** Positions are in metres in a map projection centred on the survey, chosen in
notebook 02 and saved as `work/02_survey/model_frame.json`. Later notebooks read that file, so
everything uses the same frame.

**The mesh has x = north.** GoFEM's mesh axes are x = north, y = east, z = down. So mesh points,
`receivers_projected.csv` and refinement regions in notebook 03 are `(north, east, z)`, and depths
are positive down. Map plots still put east to the right.

**Material ids** in the mesh: 1 air, 2 earth (inverted), 3 sea water (real ocean), 4 mantle (fixed),
5 and up thin-sheet ocean classes. An inversion replaces 2 with one new, large id per cell.

## For coders

The notebooks call the same functions as the GUIs, in `mtinv_gui/backends/` (setup) and
`mtinv_gui/plotting/backends/` (plotting). None of them need Qt. The main ones:

| module | used for |
|---|---|
| `backends.edi_io` | reading EDI headers and data (mtpy, with a text fallback) |
| `backends.edi_interp` | period interpolation |
| `backends.projection` | model frames, projecting stations and DEMs |
| `backends.topography` | downloading, reading, resampling DEMs |
| `backends.mesh_design` | suggested cell sizes and refinement shells |
| `backends.mesh` | mesh build steps: base grid, materials, topography fit, ocean, receivers, files |
| `backends.ocean` | coastlines and ocean conductivities |
| `backends.mesh_plots` | quick matplotlib views of a mesh |
| `backends.refine` | refining an inverted model |
| `plotting.backends.*` | loading models, georeferencing, plan views, sections, layers, GeoTIFFs |

The refinements themselves are plain pyGoFEM calls (`gofem.mesh_utils.refine_*`), so you can
replace them with your own. `help(function)` in a cell prints a function's documentation.